# World Cup 2026 Predictor — exploration

Load the trained models, predict a single match, and inspect simulation output.

Run this notebook from the project root (the folder containing `main.py`).

In [ ]:
import sys, os
# Ensure the project root is importable.
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from src import data_collection, data_cleaning, elo_model, poisson_model, ml_model
from src import feature_engineering as fe
from src.match_predictor import build_predictor
from src.tournament_simulator import TournamentSimulator

In [ ]:
# Build data + train models
data_collection.generate_sample_data()
clean = data_cleaning.run()
features = fe.run(clean)
elo = elo_model.train_elo(clean)
poisson = poisson_model.train_poisson(clean)
ml, test = ml_model.train_ml(features)
predictor = build_predictor(elo, poisson, ml, clean)

In [ ]:
# Predict a single match
predictor.predict_match('Brazil', 'France', neutral=True)

In [ ]:
# Evaluate models on the temporal hold-out
from src import evaluation
metrics, calib = evaluation.evaluate_all(predictor, test)
metrics

In [ ]:
# Run a (smaller) Monte-Carlo simulation and view title odds
sim = TournamentSimulator(predictor)
mc = sim.run_monte_carlo(n_sims=2000)
mc['summary'].head(12)